In [19]:
#create subarea shapefile with subbasin and field

In [20]:
import geopandas as gpd
import pandas as pd
import os
from rasterstats import zonal_stats
import numpy as np
import whitebox
from geocube.api.core import make_geocube

In [21]:
#file path
#existing, wetland loss, wetland restoration
output_path = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_NoBuffer/output"

#wetland enhancement 5m
output_path = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Enhancement_5m/output"

#wetland enhancement 15m
output_path = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Enhancement_15m/output"

#wetland enhancement 30m
output_path = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Enhancement_30m/output"

#wetland buffer 5m
output_path = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Buffer_5m/output"

#wetland buffer 15m
output_path = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Buffer_15m/output"

#wetland buffer 30m
output_path = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Buffer_30m/output"

#don't change from here for same scenario
#----------------------------------------------------------------------------------------
landuse_shp = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea/landuse.shp"
soil_shp = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea/soil.shp"
landuse_lookup_csv = r"D:\git\imWEBs\IMWEBsInterface\resources\DemoDatasets\BRC2021\landuse.csv"
soil_lookup_csv = r"D:\git\imWEBs\IMWEBsInterface\resources\DemoDatasets\BRC2021\soil.csv"

#don't change anything from here
#----------------------------------------------------------------------------------------
#id column
landuse_type_col = "LanduseTyp"
soil_type_col = "SoilTypeId"

#output files
subarea_shp = os.path.join(output_path, 'subarea.shp')
subarea_landuse_shp = os.path.join(output_path, 'subarea_landuse.shp')
subarea_soil_shp = os.path.join(output_path, 'subarea_soil.shp')
subarea_landuse_csv = os.path.join(output_path, 'SubareaLanduseType.csv')
subarea_soil_csv = os.path.join(output_path, 'SubareaSoilType.csv')

#fixed column name, don't change
subarea_id_col = "Id"
subarea_area_col = "Area"

lanuse_soil_subrea_id_col = "SubAreaId"
lanuse_soil_subarea_landuse_type_col = "LanduseTypeId"
lanuse_soil_subarea_soil_type_col = "SoilTypeId"

landuse_lookup_original_id_col = "OrigLU"
landuse_lookup_imwebs_id_col = "imWEBsCode"
soil_lookup_original_id_col = "OrigSoil"
soil_lookup_imwebs_id_col = "imWEBsSoil"

In [22]:
def identify(subarea_shp, landuse_soil_shp, subarea_landuse_soil_shp):
    """
    get subarea landuse or soil

    Parameters
    ----------
    subarea_shp             : the file path of subarea shapefile
    landuse_soil_shp        : the file path of landuse/soil shapefile 
    source_landuse_soil_col : the column name of landuse/soil type id in landuser/soil shapefile
    result_landuse_soil_col : the column name of landuse/soil type id in result csv file
    landuse_soil_csv        : the file path of result subarea landuser/soil csv file

    Returns
    -------
    DataFrame with three columns:
    1. SubAreaId - SubArea ID
    2. LanduserTypeId or SoilTypeId
    3. Area - Area in ha
    
    """
    subarea_shp_df = gpd.read_file(subarea_shp)
    landuse_soil_shp_df = gpd.read_file(landuse_soil_shp)

    #identify
    subarea_landuse_soil_shp_df = gpd.overlay(subarea_shp_df, landuse_soil_shp_df, how='identity')
    subarea_landuse_soil_shp_df.to_file(subarea_landuse_soil_shp)
    

In [23]:
def export_csv(subarea_landuse_soil_shp, source_landuse_soil_col, result_landuse_soil_col, landuse_soil_lookup_csv, lookup_original_id_col, lookup_imwebs_id_col, landuse_soil_csv):
    """
    export 

    Parameters
    ----------
    subarea_shp             : the file path of subarea shapefile
    landuse_soil_shp        : the file path of landuse/soil shapefile 
    source_landuse_soil_col : the column name of landuse/soil type id in landuser/soil shapefile
    result_landuse_soil_col : the column name of landuse/soil type id in result csv file
    landuse_soil_csv        : the file path of result subarea landuser/soil csv file

    Returns
    -------
    DataFrame with three columns:
    1. SubAreaId - SubArea ID
    2. LanduserTypeId or SoilTypeId
    3. Area - Area in ha
    
    """ 
    #read
    subarea_landuse_soil_shp_df = gpd.read_file(subarea_landuse_soil_shp)

    #get result column
    subarea_landuse_soil_shp_df[lanuse_soil_subrea_id_col] = subarea_landuse_soil_shp_df[subarea_id_col]    

    #add area in ha
    subarea_landuse_soil_shp_df[subarea_area_col] = subarea_landuse_soil_shp_df.area / 10000

    #do the lookup to conert original id to imwebs id
    landuse_soil_lookup_csv_df = pd.read_csv(landuse_soil_lookup_csv)
    landuse_soil_lookup_csv_df[source_landuse_soil_col] = landuse_soil_lookup_csv_df[lookup_original_id_col]

    subarea_landuse_soil_shp_df = subarea_landuse_soil_shp_df.merge(landuse_soil_lookup_csv_df, on = source_landuse_soil_col, how = "left")
    subarea_landuse_soil_shp_df[result_landuse_soil_col] = subarea_landuse_soil_shp_df[lookup_imwebs_id_col]
  
    #save to csv
    subarea_landuse_soil_shp_df.to_csv(landuse_soil_csv, columns=[lanuse_soil_subrea_id_col, result_landuse_soil_col, subarea_area_col], index = False)

    #fill na
    df = pd.read_csv(landuse_soil_csv)
    df[result_landuse_soil_col] = df[result_landuse_soil_col].fillna(method="ffill")
    df.to_csv(landuse_soil_csv, index = False)


In [24]:
#step 3: get subarea landuse and soil
print("get subarea landuse")
export_csv(subarea_landuse_shp, landuse_type_col, lanuse_soil_subarea_landuse_type_col, landuse_lookup_csv, landuse_lookup_original_id_col, landuse_lookup_imwebs_id_col, subarea_landuse_csv)
print("get subarea soil")
export_csv(subarea_soil_shp, soil_type_col, lanuse_soil_subarea_soil_type_col, soil_lookup_csv, soil_lookup_original_id_col, soil_lookup_imwebs_id_col, subarea_soil_csv)


get subarea landuse
get subarea soil
